라이브러리 불러오기

In [331]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import re
from sklearn.linear_model import LinearRegression

RandomForest

In [332]:
df = pd.read_csv('iphone_test.csv')
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.head()

(2176, 17)
['title', 'model_family', 'generation_number', 'is_pro', 'storage_gb', 'condition', 'condition_score', 'price', 'wasPrice', 'price_discount_pct', 'available', 'sold', 'seller', 'itemLocation', 'us_state', 'lastUpdated', 'color']
title                     str
model_family              str
generation_number     float64
is_pro                   bool
storage_gb              int64
condition                 str
condition_score         int64
price                 float64
wasPrice              float64
price_discount_pct    float64
available             float64
sold                  float64
seller                    str
itemLocation              str
us_state                  str
lastUpdated               str
color                     str
dtype: object


,title,model_family,generation_number,is_pro,storage_gb,condition,condition_score,price,wasPrice,price_discount_pct,available,sold,seller,itemLocation,us_state,lastUpdated,color
0,Apple iPhone 14 128gb RED color (Factory Unloc...,iPhone 14,14.0,False,128,Used,2,329.99,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-01-02,RED
1,Lot of 10 Apple iPhone 12 Pro 128GB Unlocked M...,iPhone 12 Pro,12.0,True,128,Used,2,3160.00,NaN,NaN,NaN,NaN,Seller 332,"Winter Park, Florida, United States",FL,2026-03-01,NaN
2,Apple iPhone 14 Pro MAX 128GB FULLY Unlocked,iPhone 14 Pro Max,14.0,True,128,Used,2,726.99,NaN,NaN,3.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12,NaN
3,Apple iPhone 14 Pro MAX 256gb Space black (Fac...,iPhone 14 Pro Max,14.0,True,256,Used,2,669.99,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12,black
4,Apple iPhone 14 Pro MAX 512gb Deep purple (Fac...,iPhone 14 Pro Max,14.0,True,512,Used,2,659.99,NaN,NaN,2.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-03-10,purple


Train/Test 분리 + 전처리 확인

In [333]:
# [c for c in df.columns if "price" in c.lower()]

In [334]:
# price_cols = [c for c in df.columns if "price" in c.lower()]
X = df.drop(columns=["price"])
y = df["price"]

print(X.shape, y.shape)
X.head()

(2176, 16) (2176,)


,title,model_family,generation_number,is_pro,storage_gb,condition,condition_score,wasPrice,price_discount_pct,available,sold,seller,itemLocation,us_state,lastUpdated,color
0,Apple iPhone 14 128gb RED color (Factory Unloc...,iPhone 14,14.0,False,128,Used,2,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-01-02,RED
1,Lot of 10 Apple iPhone 12 Pro 128GB Unlocked M...,iPhone 12 Pro,12.0,True,128,Used,2,NaN,NaN,NaN,NaN,Seller 332,"Winter Park, Florida, United States",FL,2026-03-01,NaN
2,Apple iPhone 14 Pro MAX 128GB FULLY Unlocked,iPhone 14 Pro Max,14.0,True,128,Used,2,NaN,NaN,3.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12,NaN
3,Apple iPhone 14 Pro MAX 256gb Space black (Fac...,iPhone 14 Pro Max,14.0,True,256,Used,2,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12,black
4,Apple iPhone 14 Pro MAX 512gb Deep purple (Fac...,iPhone 14 Pro Max,14.0,True,512,Used,2,NaN,NaN,2.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-03-10,purple


In [335]:
# 범주형 데이터 원핫인코딩 필요
cat_cols = X.select_dtypes(include="object").columns.tolist()
print("범주형 컬럼:", cat_cols)
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(X.shape)
X.head()

범주형 컬럼: ['title', 'model_family', 'condition', 'seller', 'itemLocation', 'us_state', 'lastUpdated', 'color']
(2176, 3305)


C:\Users\vpseb\AppData\Local\Temp\ipykernel_10592\329843962.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()


,generation_number,is_pro,storage_gb,condition_score,wasPrice,price_discount_pct,available,sold,title_(New) Apple iPhone 17 5G 8GB+256GB SAGE Dual SIM Unlocked iOS Cell Phone,title_*NEW OEM SEALED* Apple iPhone 16 - 128GB - Black (Unlocked)Apple 1yr Warranty,...,color_White,color_Yellow,color_black,color_blue,color_gold,color_navy,color_pink,color_purple,color_red,color_white
0,14.0,False,128,2,NaN,NaN,NaN,NaN,False,False,...,False,False,False,False,False,False,False,False,False,False
1,12.0,True,128,2,NaN,NaN,NaN,NaN,False,False,...,False,False,False,False,False,False,False,False,False,False
2,14.0,True,128,2,NaN,NaN,3.0,NaN,False,False,...,False,False,False,False,False,False,False,False,False,False
3,14.0,True,256,2,NaN,NaN,NaN,NaN,False,False,...,False,False,True,False,False,False,False,False,False,False
4,14.0,True,512,2,NaN,NaN,2.0,NaN,False,False,...,False,False,False,False,False,False,False,True,False,False


In [336]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(1740, 3305) (436, 3305)


In [337]:
y_train_log = np.log1p(y_train)

In [338]:
# train_test_split 직후, 모델 학습 전에 한번 확인
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

wasPrice              1640
price_discount_pct    1640
available              836
sold                  1072
dtype: int64


RandomForestRegressor 학습 + 평가 지표

In [356]:
# rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1) #: 1차 모델학습 파라미터
rf_model = RandomForestRegressor(n_estimators=200,
                                 max_depth= 11,
                                 min_samples_leaf=1,
                                  random_state=42,
                                  n_jobs=-1
                                  )
rf_model.fit(X_train, y_train_log)

train_preds_log = rf_model.predict(X_train) # 과적합 여부 판단을 위해 r2_train을 구함 
test_preds_log = rf_model.predict(X_test)

train_preds= np.expm1(train_preds_log)
test_preds= np.expm1(test_preds_log)

mae = mean_absolute_error(y_test, test_preds)
mse = mean_squared_error(y_test, test_preds)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
r2_train = r2_score(y_train,train_preds)
r2_test = r2_score(y_test, test_preds)

print("===== RandomForest =====")
print(f"RF_MAE: {mae}")
print(f"RF_MSE: {mse}")
print(f"RF_RMSE: {rmse}")
print(f"RF_R2: {r2_test:.3f}")
print(f'RF_R2_train: {r2_train:.3f}')
print(f'R2_Gap :{r2_train-r2_test:.3f}')



===== RandomForest =====
RF_MAE: 94.31746888332391
RF_MSE: 106573.81712633873
RF_RMSE: 326.45645517639673
RF_R2: 0.537
RF_R2_train: 0.758
R2_Gap :0.221


### 평가 기준

과적합 판단
- train r2와 test r2의 차이가 0.2 이상이면  과적합으로 판단


도메인별 현실적인 R² 스코어 기준
- 0.8 ~ 0.9 이상 (최상위권)
- 0.5 ~ 0.7 (우수함)
- 0.2 ~ 0.4 (수용 가능)


Train_R2_Score 기준 과소적합 판단
- -0.05 이내 하락 (미세 조정)
- -0.05 ~ -0.15 하락 (강한 규제)
- -0.15 이상 폭락 (과소적합 발생)

### 하이퍼파라미터 튜닝 실험결과

실험 1차 결과 => 기본
- RF_R2: 0.543
- RF_R2_train: 0.808
- R2_Gap :0.265
- 상태 : 과적합
- 방향성 : max_depth로 질문개수 정해주기, 기본 무제한으로 되어 있어 과한 학습이 일어남


실험 2차 결과 => max_depth = 10 추가 
- RF_R2: 0.535
- RF_R2_train: 0.736 
- R2_Gap :0.201
- 상태 : 과적합 경계선
- 방향성 : 말단노드에 남겨야 할 최소 데이터의 수를 제한(남겨야할 데이터가 많으면 학습을 포기할수 있음)


실험 3차 결과 => min_samples_leaf=2 추가
- RF_R2: 0.523
- RF_R2_train: 0.584 (훈련점수 갭 : 0.152)
- R2_Gap :0.061
- 상태: 과소적합 => 2차와의 Train_R2_Score Gap이 0.152로 과소적합 발생
- 방향성 => 과소적합 발생


실험 4차 결과 => max_depth = 11 변경
- RF_R2: 0.525
- RF_R2_train: 0.594 (훈련점수 갭 : 0.142) -> 강한 규제 상태로 보임
- R2_Gap :0.069
- 방향성 => min_samples_leaf=1 (R2_Score를 올리기 위해 규제를 풀어봄)


실험 5차 결과 => min_samples_leaf=1(규제를 풀음)
- RF_R2: 0.537
- RF_R2_train: 0.758 
- R2_Gap :0.221
- 과적합 
- 방향성 : RF_R2 값은 올라갔지만 규제를 풀자마자 과적합 발생, max_depth = 9로 변경

실험 6차 결과 => max_depth = 9 변경
- RF_R2: 0.532 (우수함)
- RF_R2_train: 0.712(훈련점수 갭 : -0.046)
- R2_Gap :0.181 (과적합 아님)
- 일반화 상태
- 최종 튜닝 파라미터 : max_depth = 9, min_samples_leaf=1


----
XGBoost

In [340]:
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
)

# ========================================================
# XGBoost가 허용하지 않는 특수문자를 언더스코어로 치환
X_train.columns = [re.sub(r"[\[\]<>]", "_", col) for col in X_train.columns]
X_test.columns = [re.sub(r"[\[\]<>]", "_", col) for col in X_test.columns]
# =========================================================

xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_mse = mean_squared_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2 = r2_score(y_test, xgb_preds)

print("===== XGBoost =====")
print(f"XGB_MAE: {xgb_mae}")
print(f"XGB_MSE: {xgb_mse}")
print(f"XGB_RMSE: {xgb_rmse}")
print(f"XGB_R2: {xgb_r2}")

===== XGBoost =====
XGB_MAE: 92.67477630125272
XGB_MSE: 114521.8140002441
XGB_RMSE: 338.41071791573637
XGB_R2: 0.502734372474141


---
LinearRegression

In [341]:
X_train.isnull().sum()[X_train.isnull().sum() > 0]

wasPrice              1640
price_discount_pct    1640
available              836
sold                  1072
dtype: int64

In [342]:
# 학습 데이터 기준으로 결측치 채우기
X_train_filled = X_train.fillna(X_train.mean())
X_test_filled = X_test.fillna(X_train.mean())  # test도 train의 평균값 기준으로 채워야 함 (data leakage 방지)

In [343]:
lr_model = LinearRegression()
lr_model.fit(X_train_filled, y_train)

lr_preds = lr_model.predict(X_test_filled)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

print("===== LinearRegression =====")
print(f"LR_MAE: {lr_mae}")
print(f"LR_RMSE: {lr_rmse}")
print(f"LR_R2: {lr_r2}")

===== LinearRegression =====
LR_MAE: 111.66184533994816
LR_RMSE: 318.7053835500621
LR_R2: 0.5589589420559136


In [344]:
# 확인해보기
print(df["model_family"].nunique())
print(df["generation_number"].nunique())

20
6


In [345]:
print(df.shape)

(2176, 17)


In [346]:
# 최종 모델 선택 (여기선 RandomForest 예시)
final_model = rf_model
final_preds = preds  # rf_model의 predict 결과

# 가격 범위 계산에 쓸 residual 표준편차
residuals = y_test - final_preds
residual_std = np.std(residuals)
print(f"residual_std: {residual_std}")

# predict.py에서 인코딩을 똑같이 맞추기 위해 학습 때 쓴 컬럼 목록도 저장
feature_columns = X_train.columns.tolist()

residual_std: 320.73508579367535


In [347]:
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump({
    "model": final_model,
    "feature_columns": feature_columns,
    "residual_std": residual_std,
}, "../models/price_model.pkl")

print("저장 완료: models/price_model.pkl")

저장 완료: models/price_model.pkl
